# This notebook is used to preprocess fMRI data using fmriprep

### Make sure you have run S00_ProjectSetup.ipynb before starting this 
* Note: rerunning S00 won't hurt anything if youre not sure if this has already been done

In [1]:
import pandas as pd
import numpy as np
import os.path as osp 
import os
import shutil

## SELECT PRJ AND DATA FOLDERS

In [2]:
PRJ = 'test_preproc'
DATA_DIR = '/project/mdrosenberg/Wang_etal'
task = 'rest'

In [4]:
PRJ_DIR = osp.join('/project/mdrosenberg/IG/',PRJ)
fmriprep_script = osp.join(PRJ_DIR, 'code/bash/S01_fmriprep.sh')

In [5]:
SBATCH_FILE = osp.join(PRJ_DIR,'code/SBATCH/S01_fmriprep.SBATCH.sh')

In [6]:
def get_sub_list_fmriprep(DATA_DIR):
    import os

    # Get the list of files in the current directory
    files = os.listdir(DATA_DIR)
    
    # Sort the files by modification time
    sorted_files = sorted(files)
    
    sub_list = []

    for i, filename in enumerate(sorted_files):
        if filename[0:3] == 'sub':
            sub_list.append((filename))
    return sub_list

In [11]:
sub_list = get_sub_list_fmriprep(DATA_DIR)
sub_list

['sub-117',
 'sub-118',
 'sub-119',
 'sub-120',
 'sub-121',
 'sub-122',
 'sub-123',
 'sub-124',
 'sub-126',
 'sub-127',
 'sub-128',
 'sub-129',
 'sub-131',
 'sub-132',
 'sub-133',
 'sub-134',
 'sub-135',
 'sub-136',
 'sub-137',
 'sub-138',
 'sub-139',
 'sub-140',
 'sub-141',
 'sub-142',
 'sub-216',
 'sub-217',
 'sub-219',
 'sub-221',
 'sub-223',
 'sub-224',
 'sub-225',
 'sub-226',
 'sub-228',
 'sub-229',
 'sub-230',
 'sub-231',
 'sub-232',
 'sub-234',
 'sub-235',
 'sub-236',
 'sub-237',
 'sub-238',
 'sub-239',
 'sub-240',
 'sub-241',
 'sub-242',
 'sub-243',
 'sub-244',
 'sub-245']

In [12]:
sub_list = [sub_list[0]]

In [13]:
sub_list

['sub-117']

Loop through sub_list and write file to run fmriprep

In [14]:
count = 0
rerun_fmriprep = []
with open(SBATCH_FILE, 'w') as the_file:
    the_file.write(f'#sh {SBATCH_FILE} \n')
    for sub in sub_list:
        fmriprep_file = osp.join(PRJ_DIR, 'derivatives','fmriprep', sub, 'func', f'{sub}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz')
        if not osp.exists(fmriprep_file):
            the_file.write(f'export subj={sub} dir={PRJ_DIR} data={DATA_DIR}; sbatch {fmriprep_script} \n')
            count += 1
            rerun_fmriprep.append(sub)


To execute this step:
1. Go to the SBATCH directory
2. ```bash
   sh S01_fmriprep.SBATCH.sh 
   ```

OR (from anywhere) run the command at the stop of S01_fmriprep.SBATCH.sh

the .out and .err files will be written wherever you run the command from (they are numbered in order, so the first scan is the lowest number)

Note: sometimes fmriprep will fail if jobs are running at the exact same moment. This should be mostly fixed but rerun the block of code above to make sure all jobs complete. If not, the code below will delete the half-run folders so they can be restarted

# See which subjects failed and delete their folders so it can be rerun

In [ ]:
for sub in rerun_fmriprep:
    fmriprep_folder = osp.join(PRJ_DIR, 'derivatives', 'fmriprep', sub)
    freesurfer_folder = osp.join(PRJ_DIR, 'derivatives', 'fmriprep', 'freesurfer', sub)
    html_file = osp.join(PRJ_DIR, 'derivatives', 'fmriprep', f'{sub}.html')

    # Remove directories recursively if they exist
    for folder in [fmriprep_folder, freesurfer_folder]:
        if osp.exists(folder):
            shutil.rmtree(folder)
            print(f"Deleted folder: {folder}")

    # Remove HTML report if it exists
    if osp.exists(html_file):
        os.remove(html_file)
        print(f"Deleted file: {html_file}")